# 관측형 GTFS 경로탐색 — 서울 동대문구 (r5py)

실시간 도착정보 API로 구축한 **관측형 GTFS**를 다중경로 탐색엔진(**r5py / R5**)에 넣어,
동대문구 내 **출발지 → 도착지** 여정을 탐색·지도화한다. 생성된 지도를 캡처해 포스터에 활용.

**실행 환경**
- r5py(JDK21 필요) 설치 환경에서 실행 — 로컬 conda env 예: `r5py`, `JAVA_HOME=.../jdk21`.
- 서버에서 수집한 GTFS를 쓰려면: 서버에서 `gtfs_output/<날짜>/`를 push → 로컬에서 pull.
- 서버의 해당 서비스일 피드는 최신 코드로 `reprocess <날짜>` 하여 R5 호환(빈 stop_id 제거) 상태여야 함.
- 서울 OSM(pbf)은 없으면 셀1이 BBBike에서 자동 다운로드(~52MB).


In [ ]:
import os
# ── Java 환경 (r5py = JDK21). r5py import 전에 설정 ─────────────────────────
os.environ.setdefault("JAVA_HOME", "/Users/chaelyn/miniforge3/envs/jdk21/lib/jvm")  # 환경에 맞게 수정

from pathlib import Path
import urllib.request

ROOT = Path.cwd()
if not (ROOT / "gtfs_output").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
print("ROOT =", ROOT)

# ── 설정 (여기만 수정) ──────────────────────────────────────────────────────
SERVICE_DATE = "20260915"                              # 관측 GTFS 서비스일
ORIGIN       = ("신설동역", 37.57615,   127.0250446)    # (이름, 위도, 경도) — 동대문구 내
DESTINATION  = ("중랑교",  37.5914246, 127.0668384)     # (이름, 위도, 경도) — 동대문구 내
DEPARTURE    = "20:00:00"                               # 서비스일 기준 출발 시각
# 팁: 파일럿은 3개 노선(271·272·130)만 있으므로, 출발/도착은 이 노선들이 지나는
#     동대문구 정류장 부근으로 잡아야 대중교통 경로가 탐색됩니다.
# ───────────────────────────────────────────────────────────────────────────

GTFS_ZIP = ROOT / f"gtfs_output/{SERVICE_DATE}/seoul_dongdaemun_gtfs_{SERVICE_DATE}.zip"
assert GTFS_ZIP.exists(), f"GTFS zip 없음: {GTFS_ZIP}  (해당 서비스일 수집·생성 필요)"

# 서울 OSM (없으면 BBBike에서 자동 다운로드, ~52MB)
OSM_PBF = ROOT / "data/osm/Seoul.osm.pbf"
OSM_PBF.parent.mkdir(parents=True, exist_ok=True)
if (not OSM_PBF.exists()) or OSM_PBF.stat().st_size < 40_000_000:
    print("서울 OSM pbf 다운로드 중… (~52MB, 1회)")
    urllib.request.urlretrieve("https://download.bbbike.org/osm/bbbike/Seoul/Seoul.osm.pbf", OSM_PBF)
print("GTFS =", GTFS_ZIP.name, "| OSM =", OSM_PBF.name)
print(f"{ORIGIN[0]} → {DESTINATION[0]} | 출발 {SERVICE_DATE} {DEPARTURE}")

In [ ]:
%%time
# 관측형 GTFS + 서울 도로망으로 R5 네트워크 빌드 → 상세 경로 탐색
import datetime
import geopandas as gpd
from shapely.geometry import Point
from r5py import TransportNetwork, DetailedItineraries, TransportMode

net = TransportNetwork(str(OSM_PBF), [str(GTFS_ZIP)])
o = gpd.GeoDataFrame({"id": [ORIGIN[0]]},      geometry=[Point(ORIGIN[2], ORIGIN[1])],           crs="EPSG:4326")
d = gpd.GeoDataFrame({"id": [DESTINATION[0]]}, geometry=[Point(DESTINATION[2], DESTINATION[1])], crs="EPSG:4326")
dep = datetime.datetime.strptime(f"{SERVICE_DATE} {DEPARTURE}", "%Y%m%d %H:%M:%S")

di = DetailedItineraries(net, origins=o, destinations=d, departure=dep,
        transport_modes=[TransportMode.TRANSIT, TransportMode.WALK], snap_to_network=True).copy()

# 여러 대안(option) 중, 버스를 포함하는 최단 여정 선택
di["tt_min"]   = di["travel_time"].dt.total_seconds() / 60
di["wait_min"] = di["wait_time"].dt.total_seconds() / 60
opt = (di.groupby("option")
         .agg(ride=("tt_min", "sum"), wait=("wait_min", "sum"),
              has_bus=("transport_mode", lambda s: s.astype(str).str.contains("BUS").any()))
         .reset_index())
opt["trip_min"] = opt.ride + opt.wait
cand = opt[opt.has_bus].sort_values("trip_min")
best = int((cand if not cand.empty else opt.sort_values("trip_min")).iloc[0]["option"])
journey = di[di.option == best].sort_values("segment").reset_index(drop=True)
print(f"선택 여정 option={best} | 총 {opt.set_index('option').loc[best,'trip_min']:.0f}분 "
      f"(이동 {journey.tt_min.sum():.0f} + 대기 {journey.wait_min.sum():.0f}) | 구간 {len(journey)}개")

In [ ]:
# 구간표 — 수단·승하차 정류장·시각
import io, zipfile
import pandas as pd

with zipfile.ZipFile(GTFS_ZIP) as z:
    _rt = pd.read_csv(io.BytesIO(z.read("routes.txt")), dtype=str)
    _sp = pd.read_csv(io.BytesIO(z.read("stops.txt")),  dtype=str)
RN = dict(zip(_rt.route_id, _rt.route_short_name))
SN = dict(zip(_sp.stop_id, _sp.stop_name))

rows = []
for _, g in journey.iterrows():
    walk = "WALK" in str(g["transport_mode"])
    dt = g["departure_time"]
    board = "" if pd.isna(dt) else pd.to_datetime(dt).strftime("%H:%M")
    arr   = "" if pd.isna(dt) else (pd.to_datetime(dt) + g["travel_time"]).strftime("%H:%M")
    rows.append({
        "구간": int(g["segment"]),
        "수단": "도보" if walk else f"버스 {RN.get(str(g['route_id']), g['route_id'])}",
        "승차정류장": "" if walk else SN.get(str(g["start_stop_id"]), g["start_stop_id"]),
        "하차정류장": "" if walk else SN.get(str(g["end_stop_id"]),   g["end_stop_id"]),
        "출발": board, "대기(분)": round(g["wait_min"], 1),
        "이동(분)": round(g["tt_min"], 1), "도착": arr, "거리(m)": int(g["distance"]),
    })
pd.DataFrame(rows)

In [ ]:
# folium 지도 — 여정 경로(버스=색, 도보=점선)·승하차 정류장·출발/도착
import folium

ctr = [(ORIGIN[1] + DESTINATION[1]) / 2, (ORIGIN[2] + DESTINATION[2]) / 2]
m = folium.Map(location=ctr, zoom_start=13, tiles="OpenStreetMap")
COLORS = ["#e6194B", "#4363d8", "#3cb44b", "#f58231", "#911eb4"]
bi = 0
for _, g in journey.iterrows():
    if g["geometry"] is None:
        continue
    coords = [(y, x) for x, y in g["geometry"].coords]     # (lon,lat) → (lat,lon)
    if "WALK" in str(g["transport_mode"]):
        folium.PolyLine(coords, color="#666", weight=3, dash_array="6", opacity=0.8).add_to(m)
    else:
        c = COLORS[bi % len(COLORS)]; bi += 1
        rn = RN.get(str(g["route_id"]), g["route_id"])
        folium.PolyLine(coords, color=c, weight=6, opacity=0.9, tooltip=f"버스 {rn}").add_to(m)
        folium.CircleMarker(coords[0], radius=5, color=c, fill=True, fill_opacity=1,
            popup=f"승차 [{rn}] {SN.get(str(g['start_stop_id']), '')}").add_to(m)
        folium.CircleMarker(coords[-1], radius=5, color=c, fill=True, fill_opacity=1,
            popup=f"하차 [{rn}] {SN.get(str(g['end_stop_id']), '')}").add_to(m)
folium.Marker([ORIGIN[1], ORIGIN[2]], tooltip=f"출발: {ORIGIN[0]}",
              icon=folium.Icon(color="green", icon="play")).add_to(m)
folium.Marker([DESTINATION[1], DESTINATION[2]], tooltip=f"도착: {DESTINATION[0]}",
              icon=folium.Icon(color="red", icon="stop")).add_to(m)
title = ("<div style='position:fixed;top:12px;left:60px;z-index:9999;background:white;"
         "padding:8px 12px;border:1px solid #999;border-radius:6px;font-family:sans-serif;font-size:13px;"
         "box-shadow:2px 2px 6px rgba(0,0,0,.2)'>"
         f"<b>관측형 GTFS 경로탐색 (r5py)</b><br>{ORIGIN[0]} → {DESTINATION[0]} · "
         f"{SERVICE_DATE} {DEPARTURE} 출발</div>")
m.get_root().html.add_child(folium.Element(title))
out = ROOT / f"gtfs_output/{SERVICE_DATE}/journey_{ORIGIN[0]}_{DESTINATION[0]}.html"
m.save(str(out)); print("저장:", out, "  ← 브라우저로 열어 캡처(포스터용)")
m